# Statistiques descriptives — VélibData

**Tâche T8** (cf. `docs/bloc2/planning.md`) — Responsable : Lucas (Data Analyst)

Objectif : livrer des statistiques descriptives sur la performance des stations Vélib' et un contrôle qualité fonctionnel des données livrées par le pipeline (Kafka → Spark → PostgreSQL, couche silver).

Connexion : lecture directe de la base `velibdata` (tables `stations` et `disponibilite_releve`), en lecture seule.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(Path('../../.env'))

engine = create_engine(
    f"postgresql+pg8000://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ.get('POSTGRES_HOST', 'localhost')}:{os.environ.get('POSTGRES_PORT', '5432')}"
    f"/{os.environ['POSTGRES_DB']}"
)

stations = pd.read_sql('SELECT * FROM stations', engine)
disponibilite = pd.read_sql('SELECT * FROM disponibilite_releve', engine)

print(f"{len(stations)} stations, {len(disponibilite)} relevés de disponibilité")
print(f"Période couverte : {disponibilite['observed_at'].min()} → {disponibilite['observed_at'].max()}")

## 1. Vue d'ensemble du parc

In [ ]:
capacite_totale = stations['capacity'].sum()
nb_stations = len(stations)
nb_communes = stations['commune'].nunique()

print(f'Nombre de stations actives : {nb_stations}')
print(f'Capacité totale du réseau : {capacite_totale} places')
print(f'Nombre de communes couvertes : {nb_communes}')

stations.groupby('commune').size().sort_values(ascending=False).head(10)

## 2. Taux d'occupation par station

In [ ]:
df = disponibilite.merge(
    stations[['station_id', 'name', 'commune', 'capacity']], on='station_id', how='left'
)
denom = (df['num_bikes_available'] + df['num_docks_available']).replace(0, pd.NA)
df['taux_occupation'] = df['num_bikes_available'] / denom

occupation_par_station = (
    df.groupby(['station_id', 'name'])['taux_occupation']
    .mean()
    .reset_index()
    .sort_values('taux_occupation', ascending=False)
)

print("Top 10 stations les plus chargées (taux d'occupation moyen) :")
display(occupation_par_station.head(10))

print("\nTop 10 stations les moins chargées :")
display(occupation_par_station.tail(10))

## 3. Distribution du nombre de vélos disponibles

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df['num_bikes_available'].plot(kind='hist', bins=30, ax=ax, color='#2563eb', edgecolor='white')
ax.set_xlabel('Nombre de vélos disponibles par relevé')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution du nombre de vélos disponibles (tous relevés)')
plt.tight_layout()
plt.show()

## 4. Évolution temporelle agrégée (disponibilité moyenne du réseau)

In [ ]:
df['observed_at'] = pd.to_datetime(df['observed_at'])
serie_horaire = df.set_index('observed_at').resample('1h')['num_bikes_available'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
serie_horaire.plot(ax=ax, color='#16a34a')
ax.set_xlabel('Heure')
ax.set_ylabel('Vélos disponibles (moyenne réseau)')
ax.set_title('Évolution horaire de la disponibilité moyenne des vélos')
plt.tight_layout()
plt.show()

## 5. Contrôle qualité fonctionnel (cf. `pipeline_alertes`)

In [ ]:
alertes = pd.read_sql(
    "SELECT alerte_type, severity, count(*) AS nb FROM pipeline_alertes "
    "GROUP BY alerte_type, severity ORDER BY nb DESC",
    engine,
)
display(alertes)

incoherences = df[df['num_bikes_available'] + df['num_docks_available'] > df['capacity'] + 5]
print(
    f"Relevés avec un total (vélos+docks) très supérieur à la capacité affichée : "
    f"{len(incoherences)} / {len(df)}"
)

## Conclusion

Ces statistiques (taux d'occupation par station, distribution, tendance horaire, suivi des alertes qualité) alimentent directement le tableau de bord Grafana "VelibData - Usage" (T7, cf. `mspr-tech/monitoring/grafana/dashboards/velibdata-usage.json`) et servent de base au prototype de prédiction (T9, `02_prediction_demande.ipynb`).